# Merge Sort for Reporting Workloads


```{contents}
:local:
:depth: 2
```


This supporting reading adds one faster sorting algorithm to the chapter's insertion sort. It uses recursion, introduced in Chapter 5's supporting reading *Recursive Methods*; Chapter 22 develops the same algorithm with full verification and cost proofs, and Chapter 23 adds quicksort. For related explanation, see the further reading below.[^development-reading]


```{index} When Insertion Sort Is Not Enough
```

## When Insertion Sort Is Not Enough


Section 16.2 showed that insertion sort makes `n(n - 1) / 2` comparisons on reverse-ordered input. That is fine for a few hundred invoices and a problem for a nightly report over a large table:

| Records | Insertion sort, worst case | Merge sort, about `n log2 n` |
| ---: | ---: | ---: |
| 10 | 45 | 33 |
| 1,000 | 499,500 | 9,966 |
| 100,000 | 4,999,950,000 | 1,660,964 |

**Merge sort** reaches the right-hand column by never inserting one element at a time. It splits the array in half, sorts each half, and combines the two sorted halves with a step called *merge*. The halves are sorted the same way, which is where recursion comes in. The price is an extra array for the merged output, so merge sort is not an in-place sort.


```{index} Merge Two Sorted Runs
```

## Merge Two Sorted Runs


Start with the combining step, because everything else depends on it. Given two arrays that are each already sorted, called *runs*, compare their first unread elements and copy the smaller one to the output. Nothing later in either run can be smaller than that run's first unread element, so the copied value is in its final place. When one run is used up, copy the rest of the other run.

```csharp
while (i < left.Length && j < right.Length)         // both runs have unread elements
{
    if (comparer.Compare(left[i], right[j]) <= 0)    // left wins ties
        result[k++] = left[i++];
    else
        result[k++] = right[j++];
}
while (i < left.Length) result[k++] = left[i++];    // copy whichever run remains
while (j < right.Length) result[k++] = right[j++];
```

Merging the morning run `2, 5` with the afternoon run `1, 5, 7`:

| Output so far | Left unread | Right unread | Next choice |
| --- | --- | --- | --- |
| | 2, 5 | 1, 5, 7 | 1 from the right |
| 1 | 2, 5 | 5, 7 | 2 from the left |
| 1, 2 | 5 | 5, 7 | 5 from the left (tie, left wins) |
| 1, 2, 5 | | 5, 7 | copy the remaining right run |
| 1, 2, 5, 5, 7 | | | done |

The `<= 0` comparison is the stability rule from Section 16.2 in a new place: when two elements tie, the one from the left run, which came earlier in the original array, is copied first.


In [ ]:
using System;
using System.Collections.Generic;

int[] morning = { 2, 5 };
int[] afternoon = { 1, 5, 7 };
Console.WriteLine(string.Join(", ", ReportSort.Merge(morning, afternoon, Comparer<int>.Default)));
Console.WriteLine($"Empty left run: {string.Join(", ", ReportSort.Merge(Array.Empty<int>(), afternoon, Comparer<int>.Default))}");

public static class ReportSort
{
    // Both inputs must already be sorted under comparer. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, IComparer<T> comparer)
    {
        T[] result = new T[left.Length + right.Length];
        int i = 0, j = 0, k = 0;
        while (i < left.Length && j < right.Length)
        {
            if (comparer.Compare(left[i], right[j]) <= 0) result[k++] = left[i++];
            else result[k++] = right[j++];
        }
        while (i < left.Length) result[k++] = left[i++];
        while (j < right.Length) result[k++] = right[j++];
        return result;
    }
}


1, 2, 5, 5, 7
Empty left run: 1, 5, 7


Merging `n` elements in total takes at most `n - 1` comparisons and exactly `n` copies. An empty run needs no comparisons at all; the other run is copied through unchanged.


```{index} Guided Practice: Merge Descending Runs
```

### Guided Practice: Merge Descending Runs


Two branches send their amounts sorted from largest to smallest. Change one comparison so the merge produces a descending result, `9, 7, 5, 5, 2, 1`, and keep the left run winning ties.


In [ ]:
using System;

int[] north = { 9, 5, 2 };
int[] south = { 7, 5, 1 };
Console.WriteLine(string.Join(", ", MergeDescending(north, south)));

static int[] MergeDescending(int[] left, int[] right)
{
    int[] result = new int[left.Length + right.Length];
    int i = 0, j = 0, k = 0;
    while (i < left.Length && j < right.Length)
    {
        // Your code starts here.
        if (left[i] <= right[j]) result[k++] = left[i++];
        else result[k++] = right[j++];
        // Your code ends here.
    }
    while (i < left.Length) result[k++] = left[i++];
    while (j < right.Length) result[k++] = right[j++];
    return result;
}


In [ ]:
using System;

// Solution
int[] north = { 9, 5, 2 };
int[] south = { 7, 5, 1 };
Console.WriteLine(string.Join(", ", MergeDescending(north, south)));

static int[] MergeDescending(int[] left, int[] right)
{
    int[] result = new int[left.Length + right.Length];
    int i = 0, j = 0, k = 0;
    while (i < left.Length && j < right.Length)
    {
        if (left[i] >= right[j]) result[k++] = left[i++];   // larger first; left wins ties
        else result[k++] = right[j++];
    }
    while (i < left.Length) result[k++] = left[i++];
    while (j < right.Length) result[k++] = right[j++];
    return result;
}


9, 7, 5, 5, 2, 1


The starter prints `7, 5, 1, 9, 5, 2`: with an ascending comparison the merge takes the whole south run before the north run's 9 looks small enough. The `>=` keeps the tie rule, so the north 5 still precedes the south 5.


```{index} Divide, Sort Each Half, Merge
```

## Divide, Sort Each Half, Merge


Now the recursive method. The base case is an array of length zero or one, which is already sorted; return a copy so the caller always receives a fresh array. The recursive case splits the array at its middle, sorts each half with the same method, and merges the results.

```csharp
static T[] MergeSort<T>(T[] values, IComparer<T> comparer)
{
    if (values.Length <= 1) return (T[])values.Clone();   // base case
    int middle = values.Length / 2;
    T[] left = MergeSort(values[..middle], comparer);      // smaller problem
    T[] right = MergeSort(values[middle..], comparer);     // smaller problem
    return Merge(left, right, comparer);                   // combine
}
```

Each recursive call receives a strictly shorter array, so the calls reach the base case. The range expressions `values[..middle]` and `values[middle..]` create new arrays for the halves, which keeps this version easy to read; Chapter 22 replaces them with index ranges and one shared buffer. Because the method returns a new array, the arrival order survives, which is the safe reporting snapshot Section 16.2 asked for.


In [ ]:
using System;
using System.Collections.Generic;

int[] arrived = { 300, 100, 200, 100, 50 };
int[] report = ReportSort.MergeSort(arrived, Comparer<int>.Default);
Console.WriteLine("Arrival: " + string.Join(", ", arrived));
Console.WriteLine("Report:  " + string.Join(", ", report));
Console.WriteLine($"Empty input sorts to length {ReportSort.MergeSort(Array.Empty<int>(), Comparer<int>.Default).Length}");

public static class ReportSort
{
    public static T[] MergeSort<T>(T[] values, IComparer<T> comparer)
    {
        if (values.Length <= 1) return (T[])values.Clone();   // base case: already sorted
        int middle = values.Length / 2;
        T[] left = MergeSort(values[..middle], comparer);      // sort the first half
        T[] right = MergeSort(values[middle..], comparer);     // sort the second half
        return Merge(left, right, comparer);                   // combine
    }

    // Both inputs must already be sorted under comparer. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, IComparer<T> comparer)
    {
        T[] result = new T[left.Length + right.Length];
        int i = 0, j = 0, k = 0;
        while (i < left.Length && j < right.Length)
        {
            if (comparer.Compare(left[i], right[j]) <= 0) result[k++] = left[i++];
            else result[k++] = right[j++];
        }
        while (i < left.Length) result[k++] = left[i++];
        while (j < right.Length) result[k++] = right[j++];
        return result;
    }
}


Arrival: 300, 100, 200, 100, 50
Report:  50, 100, 100, 200, 300
Empty input sorts to length 0


Trace the splits for `300, 100, 200, 100, 50`: the first call splits into `300, 100` and `200, 100, 50`; those split again until every piece has one element; then merges rebuild `100, 300`, then `100, 50` becomes `50, 100`, then `200` merges with it to give `50, 100, 200`, and the final merge produces `50, 100, 100, 200, 300`. Drawing that tree of calls is the fastest way to see why the work is small: each level of the tree copies every element once, and there are only about `log2 n` levels.


**Visualize it.** OpenDSA's [Mergesort Concepts](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Mergesort.html) slideshow draws the tree of splits and merges described above, and [Implementing Mergesort](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/MergesortImpl.html) steps through the merge index by index.

```{index} Code Tracing: Follow the Recursion
```

### Code Tracing: Follow the Recursion

Trace `MergeSort` on `[5, 2, 4, 7, 1, 3]` without running it. Draw the splits as a tree: each node is an array, its two children are its halves, and the leaves are single elements. Then write the merged result at each node from the leaves upward. How many levels does the tree have, and how many merges happen in total? The solution prints each split and each merge in the order the program performs them.

In [ ]:
// Solution
using System;
using System.Collections.Generic;

int merges = 0;
int[] result = Sort(new[] { 5, 2, 4, 7, 1, 3 }, 0);
Console.WriteLine($"result [{string.Join(", ", result)}], merges = {merges}");

int[] Sort(int[] values, int depth)
{
    string indent = new string(' ', depth * 2);
    if (values.Length <= 1) { Console.WriteLine($"{indent}leaf  [{string.Join(", ", values)}]"); return values; }
    Console.WriteLine($"{indent}split [{string.Join(", ", values)}]");
    int half = values.Length / 2;
    int[] left = Sort(values[..half], depth + 1);
    int[] right = Sort(values[half..], depth + 1);
    int[] merged = new int[values.Length];
    int i = 0, j = 0, k = 0;
    while (i < left.Length && j < right.Length) merged[k++] = left[i] <= right[j] ? left[i++] : right[j++];
    while (i < left.Length) merged[k++] = left[i++];
    while (j < right.Length) merged[k++] = right[j++];
    merges++;
    Console.WriteLine($"{indent}merge [{string.Join(", ", left)}] + [{string.Join(", ", right)}] -> [{string.Join(", ", merged)}]");
    return merged;
}

split [5, 2, 4, 7, 1, 3]
  split [5, 2, 4]
    leaf  [5]
    split [2, 4]
      leaf  [2]
      leaf  [4]
    merge [2] + [4] -> [2, 4]
  merge [5] + [2, 4] -> [2, 4, 5]
  split [7, 1, 3]
    leaf  [7]
    split [1, 3]
      leaf  [1]
      leaf  [3]
    merge [1] + [3] -> [1, 3]
  merge [7] + [1, 3] -> [1, 3, 7]
merge [2, 4, 5] + [1, 3, 7] -> [1, 2, 3, 4, 5, 7]
result [1, 2, 3, 4, 5, 7], merges = 5


The tree has four levels, counting the six leaves as the deepest, and five merges happen: one per internal node, which is always one fewer than the number of elements. Notice the order of the printed lines: the program finishes the entire left half, `[5, 2, 4]`, before it looks at the right half, because each recursive call must return a sorted array before the merge above it can proceed. A trace that merges level by level from the bottom describes the same tree but not the program's actual order of work. Chapter 22 uses this tree to count the total comparisons.

```{index} Count the Work
```

## Count the Work


Add a comparison counter to `Merge` and sort reverse-ordered arrays of growing size. The counts never exceed `n log2 n` and grow at that rate, while insertion sort's worst case grows with `n` squared. On reversed input each merge exhausts one run early, so the count lands near half the bound. The same counter approach was used in Section 16.2; counting comparisons supports a growth claim in a way that one stopwatch run cannot.


In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

foreach (int n in new[] { 4, 8, 16, 32, 1024 })
{
    int[] reversed = Enumerable.Range(1, n).Reverse().ToArray();
    ReportSort.Comparisons = 0;
    ReportSort.MergeSort(reversed, Comparer<int>.Default);
    long insertionWorst = (long)n * (n - 1) / 2;
    Console.WriteLine($"n={n,5}: merge comparisons={ReportSort.Comparisons,6}  n*log2(n)={n * Math.Log2(n),7:F0}  insertion worst={insertionWorst,7}");
}

public static class ReportSort
{
    public static long Comparisons;

    public static T[] MergeSort<T>(T[] values, IComparer<T> comparer)
    {
        if (values.Length <= 1) return (T[])values.Clone();   // base case: already sorted
        int middle = values.Length / 2;
        T[] left = MergeSort(values[..middle], comparer);      // sort the first half
        T[] right = MergeSort(values[middle..], comparer);     // sort the second half
        return Merge(left, right, comparer);                   // combine
    }

    // Both inputs must already be sorted under comparer. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, IComparer<T> comparer)
    {
        T[] result = new T[left.Length + right.Length];
        int i = 0, j = 0, k = 0;
        while (i < left.Length && j < right.Length)
        {
            Comparisons++;
            if (comparer.Compare(left[i], right[j]) <= 0) result[k++] = left[i++];
            else result[k++] = right[j++];
        }
        while (i < left.Length) result[k++] = left[i++];
        while (j < right.Length) result[k++] = right[j++];
        return result;
    }
}


n=    4: merge comparisons=     4  n*log2(n)=      8  insertion worst=      6
n=    8: merge comparisons=    12  n*log2(n)=     24  insertion worst=     28
n=   16: merge comparisons=    32  n*log2(n)=     64  insertion worst=    120
n=   32: merge comparisons=    80  n*log2(n)=    160  insertion worst=    496
n= 1024: merge comparisons=  5120  n*log2(n)=  10240  insertion worst= 523776


Merge sort does the same amount of work whether the input is sorted, reversed, or random, because it always splits and always merges. Insertion sort is faster than merge sort on input that is already nearly sorted, which is why a small, mostly ordered update can still be a good insertion sort workload.


```{index} Verify Order, Preservation, and Stability
```

## Verify Order, Preservation, and Stability


Section 16.2's checks apply unchanged: the output must be ordered under the comparer, must contain exactly the input occurrences, and must keep equal keys in arrival order. The invoice comparer from that section orders by amount only, so the tie between invoices A, C, and E at 200 and between B and D at 100 tests stability. The expected order is B, D, A, C, E.


In [ ]:
using System;
using System.Collections.Generic;

Invoice[] invoices = { new Invoice("A", 200m), new Invoice("B", 100m), new Invoice("C", 200m), new Invoice("D", 100m), new Invoice("E", 200m) };
Invoice[] byAmount = ReportSort.MergeSort(invoices, new AmountComparer());
foreach (Invoice invoice in byAmount) Console.WriteLine($"{invoice.Id}: {invoice.Amount:F2}");
Console.WriteLine($"Original still starts with {invoices[0].Id}; report has {byAmount.Length} invoices");

public record Invoice(string Id, decimal Amount);

public class AmountComparer : IComparer<Invoice>
{
    public int Compare(Invoice x, Invoice y) => x.Amount.CompareTo(y.Amount);
}

public static class ReportSort
{
    public static T[] MergeSort<T>(T[] values, IComparer<T> comparer)
    {
        if (values.Length <= 1) return (T[])values.Clone();   // base case: already sorted
        int middle = values.Length / 2;
        T[] left = MergeSort(values[..middle], comparer);      // sort the first half
        T[] right = MergeSort(values[middle..], comparer);     // sort the second half
        return Merge(left, right, comparer);                   // combine
    }

    // Both inputs must already be sorted under comparer. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, IComparer<T> comparer)
    {
        T[] result = new T[left.Length + right.Length];
        int i = 0, j = 0, k = 0;
        while (i < left.Length && j < right.Length)
        {
            if (comparer.Compare(left[i], right[j]) <= 0) result[k++] = left[i++];
            else result[k++] = right[j++];
        }
        while (i < left.Length) result[k++] = left[i++];
        while (j < right.Length) result[k++] = right[j++];
        return result;
    }
}


B: 100.00
D: 100.00
A: 200.00
C: 200.00
E: 200.00
Original still starts with A; report has 5 invoices


If the merge used `<` instead of `<= 0`, C would be copied before A whenever they met in a merge, and the report would silently reorder equal invoices. Change the comparison, rerun, and watch the order of A, C, and E to see the defect that the stability check catches.


```{index} Choose for the Workload
```

## Choose for the Workload


| Property | Insertion sort (16.2) | Merge sort (this reading) | `Array.Sort` |
| --- | --- | --- | --- |
| Extra storage | None, in place | A second array | None, in place |
| Best case | `n - 1` comparisons on sorted input | About `n log2 n` | About `n log2 n` |
| Worst case | `n(n - 1) / 2` | About `n log2 n` | About `n log2 n` |
| Equal keys keep arrival order | Yes | Yes | Not promised |
| Input preserved | No, unless you sort a clone | Yes, returns a new array | No, unless you sort a clone |

For a small batch of mostly ordered updates, insertion sort is simple and fast. For a nightly report over a large table where tied amounts must keep arrival order, merge sort gives a predictable cost and a stable result. For most application code, the library sort is the first choice, with a clone when the original order matters and a secondary key in the comparer when ties must be deterministic. Chapter 22 proves the `n log2 n` bound and verifies merge sort systematically; Chapter 23 adds quicksort and compares complete search-and-sort workloads.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. Merge sort on an already sorted array still splits it all the way down to single elements. Is that wasted work, and what does it cost?

```{dropdown} Answer
It is work that insertion sort would avoid, but the cost stays n log n rather than growing, because each merge of two sorted runs that happen to be in order exhausts the left run quickly. Merge sort's cost barely depends on the input's order, which is a strength for unpredictable data and a weakness for data known to be nearly sorted.
```

2. Why is the `<= 0` comparison in `Merge` what makes the whole sort stable?

```{dropdown} Answer
Ties are the only place where the algorithm has a choice. Taking the left element on a tie always prefers the run that came earlier in the original array, and because every merge preserves that preference, equal keys end the sort in their original relative order.
```

3. Merge sort needs an extra array of n elements while insertion sort needs none. When does that difference decide the choice?

```{dropdown} Answer
When memory is the constraint, or when the array is small enough that the quadratic cost does not matter. For a few dozen records insertion sort is simpler and uses no extra memory; for a large nightly report the n log n saving dwarfs the cost of one extra array.
```

4. How many levels of splitting does merge sort perform on 1,000,000 records, and about how many comparisons does that imply?

```{dropdown} Answer
About 20 levels, since 2 to the 20th is just over a million, and each level merges a total of n elements with at most n comparisons, so about 20 million comparisons in all, against roughly 500 billion for insertion sort's worst case.
```

```{index} Practice: Merge Two Daily Files
```

## Practice: Merge Two Daily Files


Two branch offices each send their day's amounts already sorted. Complete the two copy loops that run after one file is exhausted. Expected output: `1, 2, 5, 5, 7, 9` and `3, 4, 8`.


In [ ]:
using System;
using System.Collections.Generic;

int[] north = { 2, 5, 9 };
int[] south = { 1, 5, 7 };
Console.WriteLine(string.Join(", ", MergeFiles(north, south)));
Console.WriteLine(string.Join(", ", MergeFiles(new[] { 3, 4, 8 }, Array.Empty<int>())));

static int[] MergeFiles(int[] left, int[] right)
{
    int[] result = new int[left.Length + right.Length];
    int i = 0, j = 0, k = 0;
    while (i < left.Length && j < right.Length)
    {
        if (left[i] <= right[j]) result[k++] = left[i++];
        else result[k++] = right[j++];
    }
    // Your code starts here.
    // Copy whatever remains of left, then whatever remains of right.
    // Your code ends here.
    return result;
}


In [ ]:
using System;
using System.Collections.Generic;

// Solution
int[] north = { 2, 5, 9 };
int[] south = { 1, 5, 7 };
Console.WriteLine(string.Join(", ", MergeFiles(north, south)));
Console.WriteLine(string.Join(", ", MergeFiles(new[] { 3, 4, 8 }, Array.Empty<int>())));

static int[] MergeFiles(int[] left, int[] right)
{
    int[] result = new int[left.Length + right.Length];
    int i = 0, j = 0, k = 0;
    while (i < left.Length && j < right.Length)
    {
        if (left[i] <= right[j]) result[k++] = left[i++];
        else result[k++] = right[j++];
    }
    while (i < left.Length) result[k++] = left[i++];
    while (j < right.Length) result[k++] = right[j++];
    return result;
}


1, 2, 5, 5, 7, 9
3, 4, 8


Without the copy loops the second call prints three zeros, because `result` was created full of default values and nothing was copied after the main loop ended. A length check alone would not catch that defect; the preservation check from Section 16.2 does.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Mergesort](https://opendsa-server.cs.vt.edu/ODSA/Books/Everything/html/Mergesort.html). The branch-office and invoice scenarios here are original teaching examples.
